## Classification des images selon leur orientation

Ce notebook réalise une première tâche d'analyse de données à partir des annotations générées automatiquement.

### Objectif :
Prédire l'orientation d'une image (`landscape` ou `portrait`) à partir de ses couleurs dominantes, extraites par clustering (`KMeans`).

### Étapes réalisées :
- Chargement des données annotées (`annotations.json`)
- Construction d'un DataFrame contenant :
  - Les couleurs dominantes (features)
  - L'orientation (label cible)
- Encodage des couleurs via `MultiLabelBinarizer`
- Apprentissage supervisé avec un `RandomForestClassifier`
- Évaluation des performances (accuracy, precision, recall, f1-score)

### Résultat :
Le modèle a bien appris à prédire les images au format paysage (`landscape`). Les performances sont limitées pour les images au format portrait (`portrait`), probablement en raison d'un déséquilibre des classes dans les données.

Cette classification servira de base pour identifier des préférences utilisateurs ou orienter la recommandation visuelle.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
import pandas as pd
import json

with open("../annotation/JSON/annotations.json", "r", encoding="utf-8") as f:
    annotations = json.load(f)

# Création d'un DataFrame images
df = pd.DataFrame([
    {
        "image": img,
        "orientation": "portrait" if details["metadata"]["size"][1] > details["metadata"]["size"][0] else "landscape",
        "colors": details["dominant_colors"]
    }
    for img, details in annotations.items()
])

# Encodage des couleurs (multi-label)
mlb = MultiLabelBinarizer()
colors_encoded = mlb.fit_transform(df["colors"])

X = pd.DataFrame(colors_encoded, columns=mlb.classes_)
y = df["orientation"]

# Entraînement du classificateur
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Rapport de classification
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

   landscape       0.94      1.00      0.97        15
    portrait       0.00      0.00      0.00         1

    accuracy                           0.94        16
   macro avg       0.47      0.50      0.48        16
weighted avg       0.88      0.94      0.91        16



C:\Users\cmvil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\cmvil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\cmvil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classificati

## Clustering non supervisé basé sur la couleur moyenne des images

Cette section a pour but de regrouper les images selon leur style visuel dominant, sans étiquette préalable.  
On applique un algorithme de clustering (`KMeans`) en se basant sur la couleur moyenne de chaque image.

### Étapes réalisées :
- Conversion des couleurs dominantes (hexadécimal) en RGB
- Calcul d’une couleur moyenne par image
- Application de `KMeans` pour regrouper les images par similarité colorimétrique
- Visualisation de la couleur moyenne de chaque cluster

### Utilité :
- Identifier des groupes visuels cohérents
- Enrichir le profil utilisateur avec un style de couleur dominant
- Préparer des recommandations basées sur des clusters visuels


In [15]:
import numpy as np
from sklearn.cluster import KMeans

# Convertir un hexadécimal en RGB
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# Calculer la couleur moyenne d'une image (à partir de ses couleurs dominantes)
def get_average_rgb(colors):
    if not colors:
        return [0, 0, 0]
    rgbs = [hex_to_rgb(c) for c in colors]
    r = sum([rgb[0] for rgb in rgbs]) / len(rgbs)
    g = sum([rgb[1] for rgb in rgbs]) / len(rgbs)
    b = sum([rgb[2] for rgb in rgbs]) / len(rgbs)
    return [r, g, b]

# Appliquer la transformation sur les couleurs de chaque image
df["rgb_avg"] = df["colors"].apply(get_average_rgb)

# Transformer en matrice pour clustering
X_rgb = np.array(df["rgb_avg"].tolist())

# Appliquer le clustering KMeans sur la couleur moyenne
n_clusters = 3
kmeans_rgb = KMeans(n_clusters=n_clusters, random_state=42)
df["cluster_rgb"] = kmeans_rgb.fit_predict(X_rgb)

# Afficher la répartition
print("Répartition des images par cluster :")
print(df["cluster_rgb"].value_counts().sort_index())

# Afficher la couleur moyenne de chaque cluster
print("\nCentres de couleur moyenne des clusters :")
for i, center in enumerate(kmeans_rgb.cluster_centers_):
    color_hex = '#%02x%02x%02x' % tuple(int(c) for c in center)
    print(f"Cluster {i} : {color_hex}")


Répartition des images par cluster :
cluster_rgb
0    23
1    41
2    15
Name: count, dtype: int64

Centres de couleur moyenne des clusters :
Cluster 0 : #909695
Cluster 1 : #767b7e
Cluster 2 : #63605f
